Korzystanie z narzędzi generatywnej AI w rozwiązywaniu zadań nie jest dozwolone

<img src="no_AI.png" alt="Use of AI allowed only when properly documented " width="100" height="100">

# Zadanie obowiązkowe [0-10] pkt

Użyj zbioru danych [Yeast](https://archive.ics.uci.edu/dataset/110/yeast) i powtórz kroki z ćwiczeń.


1. [0-2 pkt] Używając [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), dokonaj przeszukania przestrzeni hiperparametrów kNN, zmieniając (uargumentuj wybór zakresów):
   1. liczbę sąsiadów
   1. wagę dla sąsiadów
   1. metrykę
1. [0-1.5 pkt] Do wyszukiwania dodaj miary skuteczności m.in. dokładność (*accuracy*) precyzję (*precision*), czułość (*recall*, *sensitivity*), czy też współczynnik korelacji Matthewsa (MCC). Gdzie to możliwe, dostosuj opcje miar, żeby uwzględniały niezbalansowanie zbioru
1. [0-0.5 pkt] Skomentuj wyniki uzyskane w punktach 1 i 2. Spróbuj zinterpretować wyniki
1. [0-2 pkt] Dokonaj selekcji cech, używając [SequentialFeatureSelector](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html). Zmieniaj parametr `n_features_to_select` od dwóch do pięciu. Ile cech uzyskujemy przy opcji `auto`?
1. [0-0.5 pkt] Używając [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), sprawdź wpływ skalowania / normalizacji na wyniki (użyj m.in. `MinMaxScaler` oraz `RobustScaler`)
1. [0-1 pkt] Skomentuj wyniki uzyskane w punktach 4 i 5
1. [0-1.5 pkt] Dla najlepszej pary `n_features_to_select=2` wyryuj obszary decyzyjne. Skomentuj wyniki
1. [0-1 pkt] Czy w świetle uzyskanych wyników, kNN jest odpowiednim klasyfikatorem do tego problemu? Uzasadnij odpowiedź
  
<span style="color:red">**Uwaga:**</span> zadania bez komentarzy i wniosków zostaną ocenione na **0 punktów**.

# Realizacja zadania

### Załadowanie i wstępne przetwarzanie danych

(Zaczerpnięte z notebooka z ćwiczeń)

In [1]:
!pip install ucimlrepo

In [2]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

yeast = fetch_ucirepo(id=110)
X = yeast.data.features
y = yeast.data.targets

X.shape, len(y)

((1484, 8), 1484)

In [3]:
y.value_counts()

,count
localization_site,
CYT,463
NUC,429
MIT,244
ME3,163
ME2,51
ME1,44
EXC,35
VAC,30
POX,20


In [4]:
idx_to_drop = y[y['localization_site'] == 'ERL'].index
X = X.drop(idx_to_drop)
y = y.drop(idx_to_drop)

In [5]:
y.value_counts()

,count
localization_site,
CYT,463
NUC,429
MIT,244
ME3,163
ME2,51
ME1,44
EXC,35
VAC,30
POX,20


In [6]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_trans = label_encoder.fit_transform(y['localization_site'].values)

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y_trans, stratify=y_trans,test_size=0.1,random_state=0)

In [9]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Trening i ewaluacja modelu z domyślnymi hiperparametrami

In [10]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

knn = KNeighborsClassifier(n_jobs=-1)
knn.fit(X_train, y_train)

KNeighborsClassifier(n_jobs=-1)

In [11]:
y_pred_train = knn.predict(X_train)
y_pred_test = knn.predict(X_test)

In [12]:
print(f"{f1_score(y_train, y_pred_train, average='weighted'):.4f}")
print(f"{f1_score(y_test, y_pred_test, average='weighted'):.4f}")

0.6730
0.5320


In [13]:
print(f"{f1_score(y_train, y_pred_train, average='micro'):.4f}")
print(f"{f1_score(y_test, y_pred_test, average='micro'):.4f}")

0.6822
0.5473


In [14]:
print(f"{f1_score(y_train, y_pred_train, average='macro'):.4f}")
print(f"{f1_score(y_test, y_pred_test, average='macro'):.4f}")

0.6042
0.3823


In [15]:
from sklearn.metrics import matthews_corrcoef

print(f"{matthews_corrcoef(y_train, y_pred_train):.4f}")
print(f"{matthews_corrcoef(y_test, y_pred_test):.4f}")

0.5888
0.4062


In [16]:
knn.get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': -1,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

## Przeszukiwanie przestrzeni hiperparametrów

In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, matthews_corrcoef

param_grid = {
    'n_neighbors': [3,5,7,9,11,15,21,31,45],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'chebyshev']
}

scoring_metrics = {
    'accuracy': 'accuracy',
    'precision' : 'precision_weighted',
    'recall' : 'recall_macro',
    'mcc' : make_scorer(matthews_corrcoef),
    'f1' : 'f1_macro'
}

### Wybór hiperparametrów:
- n_neighbors - liczba sąsiadów - przeszukiwany jest szeroki zakres, od małych k, które mocno dopasowują się do danych treningowych, do dużych wartości, które dobrze uogólniają model, natomiast może wystąpić tu underfitting
- weights - wagi które określają wpływ sąsiadów na ostateczne przydzielenie próbki do klasy. Uniform przydziela równe wagi dla każdego sąsiada, natomiast distance określa wagę sąsiada odwrotnie proporcjonalnie do jego odległości od próbki.
- metric - metryki od p=1 manhattan, p=2 euclidean, oraz chebyshev dla p->inf. Zamiast tak określonych metryk można użyć metryki uogólnionej - minkowski - i manipulować wartością parametru p. Minkowski z p=1 to metryka manhattan, z p=2 (domyślnie w KNNeighborClassifier) to euklidean, itd.

In [18]:
knn_base = KNeighborsClassifier(n_jobs=-1)

grid_search = GridSearchCV(
    estimator=knn_base,
    param_grid=param_grid,
    cv=5,
    scoring=scoring_metrics,
    refit='mcc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 54 candidates, totalling 270 fits


GridSearchCV(cv=5, estimator=KNeighborsClassifier(n_jobs=-1), n_jobs=-1,
             param_grid={'metric': ['euclidean', 'manhattan', 'chebyshev'],
                         'n_neighbors': [3, 5, 7, 9, 11, 15, 21, 31, 45],
                         'weights': ['uniform', 'distance']},
             refit='mcc',
             scoring={'accuracy': 'accuracy', 'f1': 'f1_macro',
                      'mcc': make_scorer(matthews_corrcoef, response_method='predict'),
                      'precision': 'precision_weighted',
                      'recall': 'recall_macro'},
             verbose=1)

In [19]:
results_df = pd.DataFrame(grid_search.cv_results_)
best_idx = grid_search.best_index_
print("Wyniki miar dla najlepszego (według mcc) modelu z cross validation:")
print(f"Accuracy: {results_df.loc[best_idx, 'mean_test_accuracy']:.4f}")
print(f"Precision: {results_df.loc[best_idx, 'mean_test_precision']:.4f}")
print(f"Recall: {results_df.loc[best_idx, 'mean_test_recall']:.4f}")
print(f"MCC: {results_df.loc[best_idx, 'mean_test_mcc']:.4f}")
print(f"F1: {results_df.loc[best_idx, 'mean_test_f1']:.4f}")

Wyniki miar dla najlepszego (według mcc) modelu z cross validation:
Accuracy: 0.6123
Precision: 0.6101
Recall: 0.5503
MCC: 0.4962
F1: 0.5501


In [20]:
grid_search.best_params_

{'metric': 'euclidean', 'n_neighbors': 15, 'weights': 'distance'}

In [23]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
grid_search_model = grid_search.best_estimator_

y_pred_test_final = grid_search_model.predict(X_test)

print("wyniki modelu po grid_search na zbiorze testowym")
print(f"Accuracy: {accuracy_score(y_test, y_pred_test_final):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_test_final, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_test_final,average='macro'):.4f}")
print(f"MCC: {matthews_corrcoef(y_test, y_pred_test_final):.4f}")
print(f"F1: {f1_score(y_test, y_pred_test_final, average='macro'):.4f}")

wyniki modelu po grid_search na zbiorze testowym
Accuracy: 0.5811
Precision: 0.5568
Recall: 0.3733
MCC: 0.4516
F1: 0.3644


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Komentarz i interpretacja wyników

Zoptymalizowanie hiperparametrów za pomoca gridsearch przyniosło poprawę wyników modelu.
Dla metryki mcc według której był optymalizowany model, poprawa wyników dla zbioru testowego wyniosła z poziomu 0.4062 dla domyślnych parametrów do 0.4516 po optymalizacji. Z kolei wynik modelu dla metryki f1 macro nieznacznie spadł z poziomu 0.3823 do 0.3644.

Model ma najlepsze wyniki dla względnie wysokiego k=15 (domyślnie k=5), co zapobiega overfittingowi i uodparnia model na lokalny szum.
Dla hiperparametru weights, najlepsze wyniki uzyskał model wykorzystujący distace. Dla względnie dużego k, mniejsze klasy mogłyby zostać przeważone przez klasy liczniejsze, gdyby wykorzystany został uniform.
Domyślna metryka euclidean (czyli tak naprawdę minkowski dla p=2) dała lepszy rezultat niż pozostałe 2 przedstawione w ramach laboratorium.

Dla niezbalansowanego zbioru z jakim mamy tu doczynienia, najbardziej miarodajne są metryki f1 z argumentem average=macro oraz mcc. Analizując wynik tych metryk, model daje w miarę zadowalające wyniki, uwzględniając, że zbiór jest wysoce niezbalansowany oraz przewidujemy 9 rozróżnialnych klas.

## Selekcja cech z wykorzystaniem SequentialFeatureSelector

SequentialFeatureDelector z wykorzystaniem opcji auto

In [24]:
from sklearn.feature_selection import SequentialFeatureSelector

knn_sfs = KNeighborsClassifier(n_neighbors=15,weights='distance',metric='euclidean',n_jobs=-1)

sfs_auto = SequentialFeatureSelector(knn_sfs, n_features_to_select='auto', direction='forward',scoring=make_scorer(matthews_corrcoef),cv=5,n_jobs=-1)

sfs_auto.fit(X_train, y_train)

selected_mask_auto = sfs_auto.get_support()
num_selected_auto = selected_mask_auto.sum()
selected_cols_auto = X.columns[selected_mask_auto]

print(f"Liczba wybranych cech: {num_selected_auto}")
print(f"Wybrane cechy: {', '.join(selected_cols_auto)}")

Liczba wybranych cech: 4
Wybrane cechy: mcg, alm, mit, nuc


Przy wykorzystaniu opcji auto, zostały wybrane 4 cechy widoczne powyżej

In [25]:
from sklearn.base import clone

def evaluate_sfs_feature_subset(estimator, X_train_set, X_test_set, y_train_set, y_test_set, num_features,feature_names):
    model_clone = clone(estimator)

    sfs = SequentialFeatureSelector(
        estimator=model_clone,
        n_features_to_select=num_features,
        direction='forward',
        scoring=make_scorer(matthews_corrcoef),
        cv=5,
        n_jobs=-1
    )

    sfs.fit(X_train_set, y_train_set)

    selected_mask = sfs.get_support()
    selected_feature_names = feature_names[selected_mask].tolist()

    X_train_subset = sfs.transform(X_train_set)
    X_test_subset = sfs.transform(X_test_set)

    model_clone.fit(X_train_subset, y_train_set)

    y_pred = model_clone.predict(X_test_subset)
    mcc = matthews_corrcoef(y_test_set, y_pred)
    f1_macro = f1_score(y_test_set, y_pred, average='macro')

    print(f"liczba wybranych cech: ",num_features)
    print(f"wybrane cechy: {selected_feature_names}")
    print(f"mcc: {mcc:.4f}")
    print(f"f1 macro: {f1_macro:.4f}")

    return selected_feature_names, mcc, f1_macro

In [26]:
best_knn = KNeighborsClassifier(n_neighbors=15,weights='distance',metric='euclidean',n_jobs=-1)

for i in range (2,6):
  evaluate_sfs_feature_subset(best_knn,X_train,X_test,y_train,y_test,i,X.columns)

liczba wybranych cech:  2
wybrane cechy: ['mcg', 'alm']
mcc: 0.2245
f1 macro: 0.2472
liczba wybranych cech:  3
wybrane cechy: ['mcg', 'alm', 'mit']
mcc: 0.4000
f1 macro: 0.3244
liczba wybranych cech:  4
wybrane cechy: ['mcg', 'alm', 'mit', 'nuc']
mcc: 0.3917
f1 macro: 0.3224
liczba wybranych cech:  5
wybrane cechy: ['mcg', 'gvh', 'alm', 'mit', 'nuc']
mcc: 0.4580
f1 macro: 0.3734


Na podstawie otrzymanych wyników można stwierdzić, że im więcej cech uwzględnimy, tym lepsze wyniki modelu (z nieznacznym odchyleniem między 3 i 4 parametrami), ale nie jest to przyrost liniowy. Wynik dla pięciu parametrów nieznacznie polepszył wynik względem modelu wykorzystującego wszystkie parametry.

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, RobustScaler

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X,y_trans, stratify=y_trans, test_size=0.1, random_state=0)

best_knn = KNeighborsClassifier(n_neighbors=15,weights='distance',metric='euclidean',n_jobs=-1)

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

for scaler_name,scaler in scalers.items():
  pipeline = Pipeline(steps=[('scaler',scaler),('knn',best_knn)])

  pipeline.fit(X_train_raw,y_train)
  y_pred_pipe = pipeline.predict(X_test_raw)

  mcc_pipe = matthews_corrcoef(y_test,y_pred_pipe)
  f1_macro_pipe = f1_score(y_test,y_pred_pipe,average='macro')

  print(f"{scaler_name}:")
  print(f"mcc: {mcc_pipe:.4f}")
  print(f"f1 macro: {f1_macro_pipe:.4f}")
  print("\n")

StandardScaler:
mcc: 0.4516
f1 macro: 0.3644


MinMaxScaler:
mcc: 0.4228
f1 macro: 0.3593


RobustScaler:
mcc: 0.4865
f1 macro: 0.4052




Najlepsze wyniki dał pipeline wykorzystujący RobustScaler, któy poprawił wcześniejsze wyniki zarówno dla metryki mcc oraz f1 macro. Może to wskazywać na obecność outlierów w opracowywanym zbiorze - RobustScaler jest najbardziej odporny na tego typu przypadki, natomiast MinMaxScaler najmniej - co zgadza się z otrzymanymi powyżej wynikami.